# ABC Selective Attention - Structure-Sensitive Text Filtering

Kaggle Bench runner for the filtering task on the structure-sensitive text slice.


In [ ]:
from dataclasses import dataclass

import kaggle_benchmarks as kbench
import pandas as pd

from kaggle.shared import abc_selective_attention_utils as utils


In [ ]:
@dataclass
class LineSelectionAnswer:
    lines: list[int]


In [ ]:
DATASET_ROOT = utils.DEFAULT_DATASET_ROOT
CSV_PATH = DATASET_ROOT / 'selective_attention/structure_sensitive/text/structure_sensitive_text_full.csv'

df = utils.load_csv(CSV_PATH)
print(CSV_PATH)
print('rows:', len(df))
print('columns:', sorted(df.columns.tolist()))


In [ ]:
base_cols = [
    'seed',
    'family',
    'attentional_basis',
    'modality',
    'dimension',
    'variant',
    'structure_type',
    'structure_depth',
    'binding_distance',
    'serialization_style',
    'num_records',
]

optional_cols = [
    col
    for col in ['target_count', 'confound_count', 'confound_type', 'line_length_noise']
    if col in df.columns
]

filter_eval_df = df[base_cols + optional_cols + ['filter_prompt', 'gold_lines']].copy()
groupings = utils.build_text_groupings(df)

print('filter_eval_df columns:', filter_eval_df.columns.tolist())
print('groupings:', groupings)


In [ ]:
@kbench.task(store_task=False)
def single_structure_sensitive_text_filter_v1(
    llm,
    seed,
    family,
    attentional_basis,
    modality,
    dimension,
    variant,
    structure_type,
    structure_depth,
    binding_distance,
    serialization_style,
    num_records,
    filter_prompt,
    gold_lines,
    target_count=None,
    confound_count=None,
    confound_type=None,
    line_length_noise=None,
) -> dict:
    gold = utils.parse_gold_lines(gold_lines) if isinstance(gold_lines, str) else [int(x) for x in gold_lines]

    pred = None
    error = None
    error_type = None

    try:
        response = llm.prompt(filter_prompt, schema=LineSelectionAnswer)
        pred = [int(x) for x in response.lines]
    except Exception as exc:
        error_type, error = utils.classify_prompt_error(exc)

    is_correct = pred == gold
    has_error = error_type is not None
    is_failure = not is_correct
    failure_type = 'ok' if is_correct else (error_type if has_error else 'wrong_answer')

    kbench.assertions.assert_true(
        is_correct,
        expectation=f'Expected {gold}, got {pred}' + (f' | error_type={error_type} | error={error}' if error else ''),
    )

    result = {
        'task': 'structure_sensitive_text_filter_v1',
        'model': llm.model,
        'seed': int(seed),
        'family': family,
        'attentional_basis': attentional_basis,
        'modality': modality,
        'dimension': dimension,
        'variant': variant,
        'task_type': 'filtering',
        'structure_type': structure_type,
        'structure_depth': structure_depth,
        'binding_distance': binding_distance,
        'serialization_style': serialization_style,
        'num_records': int(num_records),
        'gold_lines': gold,
        'predicted_lines': pred,
        'is_correct': is_correct,
        'is_failure': is_failure,
        'has_error': has_error,
        'failure_type': failure_type,
    }

    if target_count not in (None, ''):
        result['target_count'] = int(target_count)
    if confound_count not in (None, ''):
        result['confound_count'] = int(confound_count)
    if confound_type not in (None, ''):
        result['confound_type'] = confound_type
    if line_length_noise not in (None, ''):
        result['line_length_noise'] = int(line_length_noise)
    if error_type is not None:
        result['error_type'] = error_type
    if error is not None:
        result['error'] = error

    return result


In [ ]:
@kbench.task(store_task=False)
def structure_sensitive_text_filter_dataset(llm) -> float:
    flat, summary = utils.run_dataset_single_model(
        item_task=single_structure_sensitive_text_filter_v1,
        llm=llm,
        eval_df=filter_eval_df,
        task_name='structure_sensitive_text_filter_dataset',
        n_jobs=8,
        groupings=groupings,
    )
    print('=== failures: structure_sensitive_text_filter_dataset ===')
    utils.print_failures(
        flat,
        columns=utils.available_columns(
            flat,
            [
                'seed',
                'family',
                'attentional_basis',
                'modality',
                'dimension',
                'variant',
                'structure_type',
                'structure_depth',
                'binding_distance',
                'serialization_style',
                'target_count',
                'confound_count',
                'confound_type',
                'line_length_noise',
                'num_records',
                'gold_lines',
                'predicted_lines',
                'failure_type',
                'error_type',
                'error',
            ],
        ),
    )
    return summary['accuracy'] * 100.0


In [ ]:
run = structure_sensitive_text_filter_dataset.run(kbench.llm)
run


In [ ]:
%choose structure_sensitive_text_filter_dataset
